# STAT163 · Week 7 · Joining tables

A small café keeps its data the way most databases do: in several tables, each about one
thing. To answer most questions you need two of them at once, and you put them together with
a **join**.

- **Parts 1 to 5, before the lecture** (about 50 minutes): you compute the café's revenue
  with `merge`, find out when a join adds rows that should not be there, and see which rows a
  join drops. The quiz covers these parts.
- **Parts 6 to 8, before the practice** (about 20 minutes): you compare each order of a guest
  with the guest's previous order. The quiz does not cover them.

One cell in Part 3 stops with an error on purpose, so run the cells one at a time rather
than all at once.

AI is welcome here. Ask it to explain a cell, or why your prediction was wrong.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/stat163-2026t1/week7-pre-lecture/main/data/"
orders = pd.read_csv(url + "orders.csv")
items = pd.read_csv(url + "items.csv")
prices = pd.read_csv(url + "prices.csv")

## Part 1 · One fact in one place

The three tables:

- `orders`: one row per order, with its `id`, `date`, `time`, `table` and the loyalty `card`
  of the guest, empty for a guest without a card
- `items`: one row per dish in an order: `order_id`, `dish` and `qty`, the quantity
- `prices`: the price of each dish in each `week` of the year. The café works in weeks 38
  and 39.

In [2]:
orders

,id,date,time,table,card
0,1,2026-09-17,08:15,2,1013.0
1,2,2026-09-17,08:40,5,NaN
2,3,2026-09-17,12:30,3,1042.0
3,4,2026-09-19,09:10,2,1013.0
4,5,2026-09-19,10:05,4,1101.0
5,6,2026-09-19,13:20,1,NaN
6,7,2026-09-22,08:20,2,1013.0
7,8,2026-09-22,12:45,3,1042.0
8,9,2026-09-24,08:30,6,1013.0
9,10,2026-09-24,09:50,4,1101.0


In [3]:
items

,order_id,dish,qty
0,1,latte,1
1,1,croissant,1
2,2,espresso,2
3,3,cappuccino,1
4,3,cheesecake,1
5,4,latte,1
6,5,espresso,1
7,5,water,1
8,6,cappuccino,2
9,6,croissant,2


In [4]:
prices

,dish,week,price
0,espresso,38,45
1,latte,38,70
2,cappuccino,38,65
3,croissant,38,55
4,cheesecake,38,90
5,espresso,39,45
6,latte,39,75
7,cappuccino,39,70
8,croissant,39,55
9,cheesecake,39,90


`items` has no price, and `prices` has no orders. The price of a latte in week 39 is written
once, in one row of `prices`. If the café changes it, one row changes, and every order of
that week gets the new price the next time you join the tables. If the price were copied
into every item, one change would have to be made in every copy.

A **key** is the column, or set of columns, that identifies exactly one row of a table: `id`
in `orders`. A join matches rows of two tables that have the same value of a key.

## Part 2 · Two tables joined with `merge`

The price depends on the week, and `items` does not have the date of the order. The date is
in `orders`. With `merge` you take each row of `items`, find the row of `orders` with the same
order number, and put its columns beside it. The number is called `order_id` in `items` and
`id` in `orders`, so you name the key column of each table with `left_on` and `right_on`:

In [5]:
dated = items.merge(orders, left_on="order_id", right_on="id")
dated

,order_id,dish,qty,id,date,time,table,card
0,1,latte,1,1,2026-09-17,08:15,2,1013.0
1,1,croissant,1,1,2026-09-17,08:15,2,1013.0
2,2,espresso,2,2,2026-09-17,08:40,5,NaN
3,3,cappuccino,1,3,2026-09-17,12:30,3,1042.0
4,3,cheesecake,1,3,2026-09-17,12:30,3,1042.0
5,4,latte,1,4,2026-09-19,09:10,2,1013.0
6,5,espresso,1,5,2026-09-19,10:05,4,1101.0
7,5,water,1,5,2026-09-19,10:05,4,1101.0
8,6,cappuccino,2,6,2026-09-19,13:20,1,NaN
9,6,croissant,2,6,2026-09-19,13:20,1,NaN


`items` is the **left** table, the one you call `merge` on, and `orders` is the **right**
table, the one inside `merge(...)`. Both key columns stay in the result: `order_id` and `id`
hold the same values. `card` shows `1013.0`: the column has empty values, so `read_csv`
reads it as decimals.

The result has the 20 rows of `items`. Each item found exactly one row in `orders`, because
`id` appears in `orders` once. This kind of join is a **lookup**: a value that belongs to the
order is written on every item of the order. When the key column has the same name in both
tables, you name it once with `on=`.

SQL, dplyr and most writing about data call this operation a join. pandas calls the method
`merge`, as base R does with its function `merge()`. pandas also has a `join` method for a
narrower job: with it you match the rows of the left table to the **index** of the right
table, its row labels, never to a column. `items.join(orders)` gives no error, and the rows
are paired by their numbers 0, 1, 2…, so the croissant of order 1 gets the date of order 2.
For joins on columns, use `merge`.

## Part 3 · A join that adds rows

To compute the revenue, every item needs its price. Both `items` and `prices` have a column
`dish`.

**Predict: how many rows does `items.merge(prices, on="dish")` return? `items` has 20, and two
of them are tap water, which is free and has no price.**

In [ ]:
priced = items.merge(prices, on="dish")
len(priced)

<details>
<summary>Answer</summary>

**36 rows**. `dish` is not a key of `prices`: each dish appears there twice, once per week. So
each item with a price finds two rows and appears twice in the result, once with
each week's price. This is called **fan-out**. There is no error and no warning: pandas sees nothing
wrong.

The two glasses of water are gone. By default `merge` keeps only the rows that found a match
in both tables, and water has no row in `prices`.

</details>

Look at the first order, a latte and a croissant:

In [7]:
priced[priced["order_id"] == 1]

,order_id,dish,qty,week,price
0,1,latte,1,38,70
1,1,latte,1,39,75
2,1,croissant,1,38,55
3,1,croissant,1,39,55


Every sum you compute on `priced` now counts each item twice, once at each week's price:

In [8]:
priced["revenue"] = priced["qty"] * priced["price"]
priced["revenue"].sum()

np.int64(2740)

To catch this before it reaches a number, state what you expect. With
`validate="many_to_one"` you say: many rows of the left table may share a key value, but on
the right each key value appears once. If that is false, the cell stops:

In [9]:
items.merge(prices, on="dish", validate="many_to_one")

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
       dish
  espresso
     latte
cappuccino
 croissant
cheesecake ...

`Merge keys are not unique in right dataset` means that a value of `dish` appears more than
once in `prices`. The other options name the other cases: `"one_to_one"`, each key value once
on both sides; `"one_to_many"`, once on the left; `"many_to_many"`, no check.

## Part 4 · A key of two columns

The key of `prices` is two columns: `dish` and `week` together name one row. To join on both,
the items need the week of their order. With `.dt.isocalendar().week` you get the week of the
year from a date. Weeks start on Monday, so Monday 21 September opens week 39:

In [10]:
dated["week"] = pd.to_datetime(dated["date"]).dt.isocalendar().week
dated[["order_id", "date", "week", "dish"]]

,order_id,date,week,dish
0,1,2026-09-17,38,latte
1,1,2026-09-17,38,croissant
2,2,2026-09-17,38,espresso
3,3,2026-09-17,38,cappuccino
4,3,2026-09-17,38,cheesecake
5,4,2026-09-19,38,latte
6,5,2026-09-19,38,espresso
7,5,2026-09-19,38,water
8,6,2026-09-19,38,cappuccino
9,6,2026-09-19,38,croissant


**Predict: how many rows does the join on both columns return?**

In [ ]:
priced = dated.merge(prices, on=["dish", "week"], validate="many_to_one")
len(priced)

<details>
<summary>Answer</summary>

**18**, one row per item with a price. With a list in `on` you join on several columns: a row
of `prices` matches only when both `dish` and `week` are equal. Each pair appears once in
`prices`, so `validate` passes. The two glasses of water are still missing.

</details>

**Try it:** compute the revenue in each week from `priced`.

In [12]:
priced["revenue"] = priced["qty"] * priced["price"]
# revenue per week

## Part 5 · Rows with no match

Water has no price, and matcha, new in week 39, has a price that no order uses. With `how=` you
choose which rows stay:

- `how="inner"`, the default: rows with a match in both tables
- `how="left"`: every row of the left table; where there is no match, the columns of the
  right table are empty
- `how="right"`: every row of the right table
- `how="outer"`: every row of both tables

With `indicator=True` you get a column `_merge` that says where each row came from:

In [13]:
everything = dated.merge(prices, on=["dish", "week"], how="outer", indicator=True)
everything["_merge"].value_counts()

_merge
both          18
left_only      2
right_only     1
Name: count, dtype: int64

`left_only` are the two glasses of water, with an empty `price`. `right_only` is the matcha,
with empty cells in every column that comes from `dated`:

In [14]:
everything[everything["_merge"] != "both"]

,order_id,dish,qty,id,date,time,table,card,week,price,_merge
18,NaN,matcha,NaN,NaN,NaN,NaN,NaN,NaN,39,85.0,right_only
19,5.0,water,1.0,5.0,2026-09-19,10:05,4.0,1101.0,38,NaN,left_only
20,11.0,water,1.0,11.0,2026-09-24,12:40,5.0,1042.0,39,NaN,left_only


Which `how` you need depends on the question:

- **The number of items the café served** needs every item: `how="left"`.
- **The revenue** needs only the items with a price: `how="inner"` gives the same total as
  `how="left"`, because a sum skips the empty prices.
- **Dishes on the price list that nobody ordered** need every price: `how="right"` or
  `"outer"`, then the rows with no order.

**Try it:** count the rows of each dish, so that matcha appears with 0. Change the `how` in
the cell below until it does. `count()` counts the values that are not empty, so a dish with
no order gets 0.

In [15]:
per_dish = dated.merge(prices, on=["dish", "week"])
per_dish.groupby("dish")["order_id"].count().sort_values()

dish
cheesecake    3
croissant     3
espresso      3
cappuccino    4
latte         5
Name: order_id, dtype: int64

Checks around a join:

- **the number of rows** before and after: `len()` of both
- **`validate=`**: which side must have each key value once
- **`indicator=True`**: how many rows found no match, on each side

# Before the practice · not on the quiz

In the practice you compare each visit of a website with the same visitor's earlier visits.
Parts 6 to 8 give you the tools for it on the café's orders.

## Part 6 · The previous order with `shift`

We keep the orders of guests with a loyalty card, the only orders we can follow from one
visit to the next. In a new column, `when`, we put the date and the time of an order into one
value, so that the orders of one guest can be sorted in time:

In [16]:
regulars = orders.dropna(subset=["card"]).copy()
regulars["when"] = pd.to_datetime(regulars["date"] + " " + regulars["time"])
regulars = regulars.sort_values(["card", "when"])
regulars

,id,date,time,table,card,when
0,1,2026-09-17,08:15,2,1013.0,2026-09-17 08:15:00
3,4,2026-09-19,09:10,2,1013.0,2026-09-19 09:10:00
6,7,2026-09-22,08:20,2,1013.0,2026-09-22 08:20:00
8,9,2026-09-24,08:30,6,1013.0,2026-09-24 08:30:00
2,3,2026-09-17,12:30,3,1042.0,2026-09-17 12:30:00
7,8,2026-09-22,12:45,3,1042.0,2026-09-22 12:45:00
10,11,2026-09-24,12:40,5,1042.0,2026-09-24 12:40:00
4,5,2026-09-19,10:05,4,1101.0,2026-09-19 10:05:00
9,10,2026-09-24,09:50,4,1101.0,2026-09-24 09:50:00


With `.copy()` you make `regulars` a table of its own. Without it, pandas can warn when you
add a column, because it cannot tell whether you also meant to change `orders`.

`regulars` is sorted by card and time. With `shift()` you move the values of a column one row
down: every row gets the value of the row above. With `groupby("card")` first, the shift
stays inside each guest.

**Predict: what does the first order of each guest get in `when_prev`?**

In [ ]:
regulars["when_prev"] = regulars.groupby("card")["when"].shift()
regulars[["card", "when", "when_prev"]]

<details>
<summary>Answer</summary>

A missing value, written `NaT` (not a time) for dates. The first order has no row above it
in its group, so there is nothing to take.

</details>

The **previous value** is a new kind of column: with it you compare a row with the row
before it. The difference of two times is a duration:

In [18]:
regulars["gap"] = regulars["when"] - regulars["when_prev"]
regulars.groupby("card")["gap"].median()

card
1013.0   2 days 00:55:00
1042.0   3 days 12:05:00
1101.0   4 days 23:45:00
Name: gap, dtype: timedelta64[us]

**Try it:** count the orders where the guest sits at the same table as on the previous visit.
Shift the column `table` the same way and count the rows where the two are equal.

In [19]:
# regulars["table_prev"] = ...

## Part 7 · The number of each order with `rank`

With `rank()` you number the values of a column from the smallest, 1 for the smallest. With
`groupby("card")` first, the numbering starts again for each guest:

In [20]:
regulars["n"] = regulars.groupby("card")["when"].rank()
regulars[["card", "when", "n"]]

,card,when,n
0,1013.0,2026-09-17 08:15:00,1.0
3,1013.0,2026-09-19 09:10:00,2.0
6,1013.0,2026-09-22 08:20:00,3.0
8,1013.0,2026-09-24 08:30:00,4.0
2,1042.0,2026-09-17 12:30:00,1.0
7,1042.0,2026-09-22 12:45:00,2.0
10,1042.0,2026-09-24 12:40:00,3.0
4,1101.0,2026-09-19 10:05:00,1.0
9,1101.0,2026-09-24 09:50:00,2.0


The ranks are decimals, such as `1.0`, because two equal values share a rank: a guest's first
two orders, if they were at the same minute, would both get `1.5`. With `rank(method="first")`
you give them 1 and 2 in the order of the table.

The rows with `n == 1` are the first order of each guest: one row per guest.

In [21]:
first_orders = regulars[regulars["n"] == 1][["card", "when"]]
first_orders

,card,when
0,1013.0,2026-09-17 08:15:00
2,1042.0,2026-09-17 12:30:00
4,1101.0,2026-09-19 10:05:00


**Try it:** find the last order of each guest. With `rank(ascending=False)` you number from
the largest value. Store it in a new column, `n_last`, because Part 8 needs `n`.

In [22]:
# the last order of each guest

## Part 8 · A table joined with itself

The previous order is also a row of `regulars`, so you can get it with a join. For each
order, the previous one belongs to the same guest and has `n` one smaller. Make a copy with
the columns you need and add 1 to its `n`. Order 2 in the copy then has `n` equal to 3, so a
join on `card` and `n` puts it beside order 3.

In [23]:
previous = regulars[["card", "n", "table"]].copy()
previous["n"] = previous["n"] + 1

with_previous = regulars.merge(previous, on=["card", "n"], how="left", suffixes=("", "_before"))
with_previous[["card", "n", "table", "table_before"]]

,card,n,table,table_before
0,1013.0,1.0,2,NaN
1,1013.0,2.0,2,2.0
2,1013.0,3.0,2,2.0
3,1013.0,4.0,6,2.0
4,1042.0,1.0,3,NaN
5,1042.0,2.0,3,3.0
6,1042.0,3.0,5,3.0
7,1101.0,1.0,4,NaN
8,1101.0,2.0,4,4.0


`regulars` and `previous` both have a column called `table`. With `suffixes=("", "_before")`
the column of the left table keeps its name, and the column of the copy gets `_before` at the
end. Without `suffixes`, pandas would name them `table_x` and `table_y`.

**Predict: does `with_previous` have the same number of rows as `regulars`?**

In [ ]:
print(len(regulars))
print(len(with_previous))

<details>
<summary>Answer</summary>

Yes, 9 and 9. `how="left"` keeps the first order of each guest, which finds no previous
order. Each pair of `card` and `n` appears once in `previous`, so each order finds at most
one previous order.

</details>

A **self-join** is a join of a table with itself. You use it when a rule about the key tells
you which row to compare with: the order one number earlier, an order on the same day, or a
dish in the same order.

## The same operations in SQL and in R

If you have written joins in SQL or with dplyr in R, this table shows the same operations in
pandas. `a` and `b` are two tables, and `k` is the key.

| What you do | pandas | R (dplyr) | SQL |
|---|---|---|---|
| Keep rows with a match in both tables | `a.merge(b, on="k")` | `inner_join(a, b, by = "k")` | `a JOIN b USING (k)` |
| Keep every row of the left table | `a.merge(b, on="k", how="left")` | `left_join(a, b, by = "k")` | `a LEFT JOIN b USING (k)` |
| Keep every row of the right table | `how="right"` | `right_join(a, b, by = "k")` | `a RIGHT JOIN b USING (k)` |
| Keep every row of both tables | `how="outer"` | `full_join(a, b, by = "k")` | `a FULL JOIN b USING (k)` |
| Join on key columns with different names | `left_on="order_id", right_on="id"` | `by = c("order_id" = "id")` | `ON a.order_id = b.id` |
| Join on a key of two columns | `on=["dish", "week"]` | `by = c("dish", "week")` | `USING (dish, week)` |
| Find the rows of `a` with no match | `indicator=True`, then the rows with `left_only` | `anti_join(a, b, by = "k")` | `a LEFT JOIN b ON a.k = b.k WHERE b.k IS NULL` |
| Make sure each key value appears once in `b` | `validate="many_to_one"` | `relationship = "many-to-one"` | a `PRIMARY KEY` or `UNIQUE` constraint on `b.k` |
| Name the columns that both tables have | `suffixes=("_a", "_b")` | `suffix = c("_a", "_b")` | `a.col AS col_a, b.col AS col_b` |
| Keep rows by a condition | `a[a["qty"] > 1]` | `filter(a, qty > 1)` | `WHERE qty > 1` |
| Take the previous value in the group | `groupby("card")["when"].shift()` | `group_by(card) %>% mutate(when_prev = lag(when))` | `LAG(when) OVER (PARTITION BY card ORDER BY when)` |
| Number the rows of each group in time order | `groupby("card")["when"].rank(method="first")` | `group_by(card) %>% mutate(n = row_number(when))` | `ROW_NUMBER() OVER (PARTITION BY card ORDER BY when)` |
| Join a table with itself | `regulars.merge(previous, on=["card", "n"], how="left")` | `left_join(regulars, previous, by = c("card", "n"))` | `FROM regulars r LEFT JOIN regulars p ON p.card = r.card AND p.n = r.n - 1` |

With `a.filter()` in pandas you pick columns or rows by their names, never by a condition,
so it does a different job from `filter()` in dplyr. With `shift` in pandas and `lag` in R you
get the row above in the order of the table, so you sort the table first. In SQL you write the
order inside `OVER (...)`.